# Company Default Data

### Description:
The dataset contains information on default payments & company details of Companies in India as on March 2019.

### Usage
Default

### Format
A data frame with 1384 observations on the following 11 variables

***Num***: ID of each company

***Default***: Default payment in next month (1=yes, 0=no)

***Total_assets***: Total amount of assets owned by the company

***Total_income***: Total income a business receives before any taxes, expenses, adjustments, exemptions, or deductions are taken out

***PAT_as_%_of_total_income***: Profit per sales dollar after all expenses are deducted from sales

***PBDITA_as_%_of_total_income***: Profit before depreciation, income tax and amortization divided by Total income

***PBT_as_%_of_total_income***: Profit per sales dollar before all expenses are deducted from sales

***Cash_profit_as_%_of_total_income***: Profit recorded by a business that uses the cash basis of accounting (Cash Profit:Total Cash profit)

***Current_ratio***: Liquidity ratio that measures a company's ability to pay short-term obligations or those due within one year (Current assets divided by current liabilities)

***Debt_to_equity_ratio***: Leverage ratio that calculates the weight of total debt and financial liabilities against total shareholders’ equity


### Source:
Simulated data

#### Importing the libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns # for making plots with seaborn
color = sns.color_palette()
import sklearn.metrics as metrics

import warnings
warnings.filterwarnings("ignore")

#### Importing the dataset

In [ ]:
Default = pd.read_csv('CompanyDefault.csv')

#Glimpse of Data
Default.head()

#### Fixing messy column names (containing spaces) for ease of use

In [ ]:
Default.columns = Default.columns.str.replace('as_%_of', 'to')

In [ ]:
Default.head()

#### First, let us check the number of rows (observations) and the number of columns (variables).

In [ ]:
print('The number of rows (observations) is',Default.shape[0],'\n''The number of columns (variables) is',Default.shape[1])

#### Data types of all variables

In [ ]:
Default.info()

In [ ]:
Default.duplicated().sum()

In [ ]:
Default.drop('Num', axis = 1, inplace = True)

#### Now, let us check the basic measures of descriptive statistics for the continuous variables.

In [ ]:
Default.describe()

#### Now, let us check the basic measures of descriptive statistics for the categorical variables.

In [ ]:
Default["default"].value_counts()

#### Checking proportion of default

In [ ]:
Default.default.sum() / len(Default.default)

#### Check for missing values

In [ ]:
Default.isnull().sum()

There are no missing values in the dataset.

#### Getting Top 5 rows

In [ ]:
Default.head()

In [ ]:
Default.boxplot()

In [ ]:
Default_X = Default.drop('default', axis = 1)
Default_Y = Default['default']

In [ ]:
def remove_outlier(col):
    sorted(col)
    Q1,Q3=np.percentile(col,[25,75])
    IQR=Q3-Q1
    lower_range= Q1-(1.5 * IQR)
    upper_range= Q3+(1.5 * IQR)
    return lower_range, upper_range

In [ ]:
for column in Default_X.columns:
    lr,ur=remove_outlier(Default[column])
    Default_X[column]=np.where(Default_X[column]>ur,ur,Default_X[column])
    Default_X[column]=np.where(Default_X[column]<lr,lr,Default_X[column])

In [ ]:
Default = pd.concat([Default_X, Default_Y], axis = 1)

### Correlation heatmap

In [ ]:
#calculate column correlations and make a seaborn heatmap - Before standardisation

plt.figure(figsize=(20,20))  # setting the size of figure to 12 by 10
p=sns.heatmap(Default.corr(), annot=True,cmap='coolwarm',square=True)

- Vaiables such as PAT_as_perc_of_total_income, PBDITA_as_perc_of_total_income, PBT_as_perc_of_total_income, Cash_profit_as_perc_of_total_income  are highly correlated among themselves
- Total income & Total assets seem to be highly correlated with each other

# Model Building using Logistic Regression for 'Probability at default'

## The equation of the Logistic Regression by which we predict the corresponding probabilities and then go on predict a discrete target variable is
# y = $\frac{1}{1 + {e^{-z}}}$

### Note: z  = $\beta_0$ +${\sum_{i=1}^{n}(\beta_i  X_1)}$

#### Now, Importing statsmodels modules

#### Creating logistic regression equation & storing it in f_1

model = SM.logit(formula=’Dependent Variable ~ Σ𝐼𝑛𝑑𝑒𝑝𝑒𝑛𝑑𝑒𝑛𝑡 𝑉𝑎𝑟𝑖𝑎𝑏𝑙𝑒𝑠 (𝑘)’
               data = ‘Data Frame containing the required values’).fit()

In [ ]:
import statsmodels.formula.api as SM

## Model 1

Lets check all columns we have in the dataset

In [ ]:
Default.columns

In [ ]:
f_1 = 'default ~ Total_assets + Total_income + PAT_to_total_income + PBDITA_to_total_income + PBT_to_total_income + Cash_profit_to_total_income + Current_ratio + Debt_to_equity_ratio'

#### Fitting the logistic regression model on 'Default' dataset

In [ ]:
model_1 = SM.logit(formula = f_1, data=Default).fit()

#### Checking the parameters

In [ ]:
model_1.summary()

Most of the ratio variables are insignificant. 

#### Checking the Variance Inflation Factor

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calc_vif(X):

    # Calculating VIF
    vif = pd.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    return(vif)

In [ ]:
X = Default.drop('default', axis = 1)
calc_vif(X).sort_values(by = 'VIF', ascending = False)

In [ ]:
X = X.drop('PBT_to_total_income', axis = 1)
calc_vif(X).sort_values(by = 'VIF', ascending = False)

In [ ]:
X = X.drop('Cash_profit_to_total_income', axis = 1)
calc_vif(X).sort_values(by = 'VIF', ascending = False)

In [ ]:
X = X.drop('Total_assets', axis = 1)
calc_vif(X).sort_values(by = 'VIF', ascending = False)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
y = Default['default']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 12)

In [ ]:
Default_train = pd.concat([X_train, y_train], axis = 1)
Default_test = pd.concat([X_test, y_test], axis = 1)

## Model 2

In [ ]:
model_2 = SM.logit(formula = 'default ~ PBDITA_to_total_income + PAT_to_total_income + Current_ratio + Debt_to_equity_ratio + Total_income', data=Default_train).fit()

#### Checking the coefficients

In [ ]:
model_2.summary()

PBDITA_to_total_income is an insignificant variable, therefore, we will eliminate it.

## Model 3

In [ ]:
model_3 = SM.logit(formula = 'default ~ PBDITA_to_total_income + Current_ratio + Debt_to_equity_ratio + Total_income', data=Default_train).fit()

#### Checking the coefficients

In [ ]:
model_3.summary()

Current_ratio is still insignificant, therefore, we will eliminate it

## Model 4

Adding variables pertaining to Repayment Status

In [ ]:
model_4 = SM.logit(formula = 'default ~ PBDITA_to_total_income + Debt_to_equity_ratio + Total_income', data=Default_train).fit()

In [ ]:
model_4.summary()

In [ ]:
y_prob_pred_train = model_4.predict(Default_train)

In [ ]:
y_class_pred=[]
for i in range(0,len(y_prob_pred_train)):
    if np.array(y_prob_pred_train)[i]>0.5:
        a=1
    else:
        a=0
    y_class_pred.append(a)

In [ ]:
from sklearn import metrics

In [ ]:
sns.heatmap((metrics.confusion_matrix(Default_train['default'],y_class_pred)),annot=True,fmt='.5g'
            ,cmap='Blues');
plt.xlabel('Predicted');
plt.ylabel('Actuals',rotation=0);

In [ ]:
63/229

In [ ]:
63/86

## Prediction on the Data

Now, let us see the predicted probability values.

In [ ]:
y_prob_pred_4 = model_4.predict()

In [ ]:
sns.boxplot(x=Default['default'],y=y_prob_pred_4)
plt.xlabel('Default');

#### Choosing the optimal threshold

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(Default_train['default'],y_prob_pred_train)

In [ ]:
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
optimal_threshold

#### Validating on the train set with revised threshold

In [ ]:
y_class_pred=[]
for i in range(0,len(y_prob_pred_train)):
    if np.array(y_prob_pred_train)[i]>0.26:
        a=1
    else:
        a=0
    y_class_pred.append(a)

In [ ]:
sns.heatmap((metrics.confusion_matrix(Default_train['default'],y_class_pred)),annot=True,fmt='.5g'
            ,cmap='Blues');
plt.xlabel('Predicted');
plt.ylabel('Actuals',rotation=0);

Let us now go ahead and print the classification report to check the various other parameters.

In [ ]:
print(metrics.classification_report(Default_train['default'],y_class_pred,digits=3))

#### Validating on the test set

In [ ]:
y_prob_pred_test = model_4.predict(Default_test)

In [ ]:
y_class_pred=[]
for i in range(0,len(y_prob_pred_test)):
    if np.array(y_prob_pred_test)[i]>0.26:
        a=1
    else:
        a=0
    y_class_pred.append(a)

In [ ]:
sns.heatmap((metrics.confusion_matrix(Default_test['default'],y_class_pred)),annot=True,fmt='.5g'
            ,cmap='Blues');
plt.xlabel('Predicted');
plt.ylabel('Actuals',rotation=0);

In [ ]:
print(metrics.classification_report(Default_test['default'],y_class_pred,digits=3))

# Linear Discriminant Analysis

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier

In [ ]:
LDA = LinearDiscriminantAnalysis()

In [ ]:
lda_model = LDA.fit(X_train, y_train)

In [ ]:
pred_train_lda = lda_model.predict(X_train)
pred_test_lda = lda_model.predict(X_test)

In [ ]:
print(metrics.classification_report(y_train, pred_train_lda))

In [ ]:
print(metrics.classification_report(y_test, pred_test_lda))

In [ ]:
pred_train_lda_prob = lda_model.predict_proba(X_train)[:,1]
pred_test_lda_prob = lda_model.predict_proba(X_test)[:,1]

In [ ]:
fpr, tpr, thresholds = roc_curve(y_train,pred_train_lda_prob)

In [ ]:
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
optimal_threshold

In [ ]:
y_class_pred=[]
for i in range(0,len(pred_train_lda_prob)):
    if np.array(pred_train_lda_prob)[i]>0.28:
        a=1
    else:
        a=0
    y_class_pred.append(a)

In [ ]:
sns.heatmap((metrics.confusion_matrix(y_train,y_class_pred)),annot=True,fmt='.5g'
            ,cmap='Blues');
plt.xlabel('Predicted');
plt.ylabel('Actuals',rotation=0);

In [ ]:
print(metrics.classification_report(y_train, y_class_pred,digits=3))

In [ ]:
y_class_pred=[]
for i in range(0,len(pred_test_lda_prob)):
    if np.array(pred_test_lda_prob)[i]>0.28:
        a=1
    else:
        a=0
    y_class_pred.append(a)

In [ ]:
sns.heatmap((metrics.confusion_matrix(y_test, y_class_pred)),annot=True,fmt='.5g'
            ,cmap='Blues');
plt.xlabel('Predicted');
plt.ylabel('Actuals',rotation=0);

In [ ]:
print(metrics.classification_report(y_test, y_class_pred,digits=3))

# Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [5, 10, 15],
    'min_samples_split': [15, 30, 45],
    'n_estimators': [25, 50]
}

rfcl = RandomForestClassifier()

grid_search = GridSearchCV(estimator = rfcl, param_grid = param_grid)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
best_grid = grid_search.best_estimator_

In [ ]:
pred_train_rf = best_grid.predict(X_train)
pred_test_rf = best_grid.predict(X_test)

In [ ]:
print(metrics.classification_report(y_train, pred_train_rf))

In [ ]:
print(metrics.classification_report(y_test, pred_test_rf))

### Conclusion

Amongst all the models we tried in this case Logistic Regression seems to be best aligned to our objective.

## END